# 03 — 選択したモデルの推薦を個別に確認する

## 1. Overview

- [02_analysis](02_analysis.ipynb)で保存した選択artifactからモデル・パラメータ・seedを読み込む。再入力しない。
- Test前の確認にはValidationの学習履歴だけを使用し、Testの正解を表示・調整に使わない。
- 既定の未選択状態では合成例を表示する。選択済みの最良モデルとは解釈しない。
- 問題があれば02のValidation探索かモデル実装へ戻る。確認後に記録し、02で設定を固定する。


## 2. Setup — 設定の引継ぎ

- `MUSIC_RECSYS_EXPERIMENT_CONFIG`は02・CLIと同じTOML。モデルとパラメータは`selection.json`から取得する。
- `MUSIC_RECSYS_REQUEST_ID`で対象ユーザーの要求を選択する。省略時は先頭の要求。
- 音源は同じTOMLの`[inspection.audio_files]`で指定。自動取得・自動再生しない。
- `REVIEW_ACTION=None`は確認記録を変更しない。観察後に`"accept"`または`"reject"`を明示する。
- 選択artifactがない場合のみ、従来の`MUSIC_RECSYS_DATASET_PATH`による合成・Validation入力の確認も使える。


In [ ]:
import json
import os
import tomllib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from music_recsys.data import load_dataset
from music_recsys.experiment_search import inspect_selected, load_study_config, review_selection
from music_recsys.recommendation_inspection import (
    audio_player,
    feature_comparison,
    inspect_recommendation,
    request_options,
)
from music_recsys.search_analysis import workflow_state

ROOT = Path.cwd() if Path("src").exists() else Path.cwd().parent
EXPERIMENT_CONFIG = Path(
    os.environ.get("MUSIC_RECSYS_EXPERIMENT_CONFIG", ROOT / "configs/workflow-toy.toml")
)
WORKFLOW = load_study_config(EXPERIMENT_CONFIG)
REVIEW_ACTION = None  # 明示操作: None / "accept" / "reject"
REVIEW_NOTE = ""
AUDITION_ITEM_ID = None  # 必要なら履歴曲・推薦曲のIDを指定
REQUEST_ID = os.environ.get("MUSIC_RECSYS_REQUEST_ID")
MODEL_PARAMETERS = {}
MODEL_SEED = 42
saved_dataset_path = os.environ.get("MUSIC_RECSYS_DATASET_PATH")
selected_context = None
display(workflow_state(WORKFLOW))
if (WORKFLOW.output / "selection.json").exists():
    selected_context = inspect_selected(WORKFLOW)
    dataset = selected_context["dataset"]
    MODEL = selected_context["model"]
    MODEL_PARAMETERS = selected_context["parameters"]
    MODEL_SEED = selected_context["seed"]
    DATASET_PATH = WORKFLOW.validation_dataset
    TOP_K = 20
else:
    MODEL = "popularity"
    DATASET_PATH = (
        Path(saved_dataset_path) if saved_dataset_path else ROOT / "examples/toy/dataset.json"
    )
    if json.loads(DATASET_PATH.read_text()).get("provenance", {}).get("phase") == "test":
        raise ValueError(
            "Test入力は定性的な調整に使えません。02でValidationの設定を選択してください。"
        )
    TOP_K = int(os.environ.get("MUSIC_RECSYS_TOP_K", "10" if saved_dataset_path else "2"))
    dataset = load_dataset(DATASET_PATH, top_k=TOP_K)
    print("設定は未選択：合成例／明示したValidation入力の確認です。次は02_analysisで設定選択。")
if REQUEST_ID is None:
    REQUEST_ID = dataset.requests[0].request_id if saved_dataset_path or selected_context else "r1"
settings = tomllib.loads(EXPERIMENT_CONFIG.read_text())
AUDIO_FILES = {
    k: (EXPERIMENT_CONFIG.parent / Path(v).expanduser()).resolve()
    for k, v in settings.get("inspection", {}).get("audio_files", {}).items()
}
display(pd.DataFrame(request_options(dataset)))
display(
    {
        "phase": "validation" if selected_context else "reference",
        "model": MODEL,
        "parameters": MODEL_PARAMETERS,
        "seed": MODEL_SEED,
        "top_k": TOP_K,
        "dataset": dataset.dataset_id,
        "sha256": dataset.sha256,
    }
)

## 3. Recommendation

- 選択された設定を共通の推薦実行関数へ渡す。入力は学習履歴・要求だけで、正解は渡さない。
- 順位・スコア・根拠は実装が返せるものだけを表示し、未実装のスコアを補完しない。


In [ ]:
result = inspect_recommendation(
    dataset, model=MODEL, request_id=REQUEST_ID, parameters=MODEL_PARAMETERS, seed=MODEL_SEED
)
display(
    pd.DataFrame(
        [
            {
                "model": result.model,
                "request_id": result.request.request_id,
                "user_id": result.request.user_id,
                "history_count": len(result.request.history),
                "candidate_count": len(result.request.candidate_ids),
                "returned_count": len(result.response.item_ids),
                "status": result.response.status,
            }
        ]
    )
)

## 4. Result Inspection

### ユーザーの履歴と推薦曲

| 列 | 意味 |
| --- | --- |
| `user_train_events` | このユーザーの学習期間中の再生回数 |
| `train_events` | 学習全体の再生回数 |
| `score` | Popularityの順位付けに使う再生回数。好みの確率ではない |
| `reason` | 現モデルの選択ルールに基づく根拠 |
| `in_history` | 推薦時に渡した履歴に含まれるか |

- 履歴は推薦時点の既聴曲。この表は曲単位の集計で、表示順は時系列を表さない。
- 履歴がないユーザーは空表。未聴候補がK未満なら推薦数もK未満。

In [ ]:
history_df = pd.DataFrame(result.history, columns=["item_id", "user_train_events", "train_events"])
recommendations_df = pd.DataFrame(
    result.recommendations,
    columns=["rank", "item_id", "train_events", "in_history", "score", "reason"],
)
display(history_df.rename_axis("history_row"))
with pd.option_context("display.max_colwidth", None):
    display(recommendations_df)

display(
    pd.DataFrame(
        [
            {
                "history_overlap": int(recommendations_df["in_history"].sum()),
                "requested": TOP_K,
                "returned": len(recommendations_df),
                "short_output": len(recommendations_df) < TOP_K,
            }
        ]
    )
)

feature_df = pd.DataFrame(
    feature_comparison(result, selected_context.get("features") if selected_context else None)
)
display(feature_df)
if not selected_context or not selected_context.get("features"):
    print("音楽的特徴は未取得。同じ出典の特徴をTOMLのexperiment.featuresで指定できます。")

### 人気度・候補との比較

- 履歴の図は学習全体の再生回数が多い上位20曲に限定。推薦曲と人気度を比較する。
- 候補表で、推薦された曲と選ばれなかった曲の差を確認する。
- Popularityは全体の人気を利用し、ユーザーの履歴は既聴曲の除外に使う。嗜好の類似度は計算しない。

In [ ]:
candidates_df = pd.DataFrame(result.candidates)
display(candidates_df)

history_plot = history_df.sort_values(["train_events", "item_id"], ascending=[False, True]).head(20)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, frame, title, color in [
    (axes[0], history_plot, "History (top 20 by training events)", "#627d98"),
    (axes[1], recommendations_df, "Recommendations", "#2a9d8f"),
]:
    if frame.empty:
        ax.text(0.5, 0.5, "No tracks", ha="center", transform=ax.transAxes)
    else:
        ax.bar(frame["item_id"], frame["train_events"], color=color)
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    ax.set_title(title)
    ax.set_xlabel("Track ID")
axes[0].set_ylabel("Training events (all users)")
fig.tight_layout()
plt.show()

### 試聴

- `AUDIO_FILES` に音源を登録し、`AUDITION_ITEM_ID` で履歴曲・推薦曲を選ぶ。
- 再生ボタンで試聴。未登録の曲は「音源未登録」と表示する。
- 音源の自動取得は行わない。合成IDへの音源割当はプレイヤーの動作確認用。

In [ ]:
AUDITION_ITEM_ID = AUDITION_ITEM_ID or (
    result.response.item_ids[0] if result.response.item_ids else None
)
if AUDITION_ITEM_ID is None:
    print("推薦曲なし。試聴する履歴曲のIDを指定できます。")
else:
    player = audio_player(AUDITION_ITEM_ID, AUDIO_FILES)
    display(
        pd.DataFrame(
            [
                {
                    "item_id": AUDITION_ITEM_ID,
                    "audio": "再生可能" if player is not None else "音源未登録",
                }
            ]
        )
    )
    if player is not None:
        display(player)

### 将来の確認項目

| 対象 | 確認したい内容 | 現在 |
| --- | --- | --- |
| 楽曲情報 | 曲名・アーティスト・タグと履歴の比較 | 推薦データに未接続 |
| LLM | 生成された推薦理由 | 未実装 |
| Agent | ツール呼出し・結果・推薦理由 | 未実装 |

- LangChain・LangGraphの採用はLLM・Agent実装時に判断する。

## 5. Summary / Next Actions

- 履歴との違い、人気度・音楽的特徴、説明の根拠、試聴した印象を記録する。特徴がなければ未取得とし、音楽的な推測を補完しない。
- 問題がある場合は`REVIEW_ACTION="reject"`を明示して記録し、[02_analysis](02_analysis.ipynb)のValidation探索かモデル実装へ戻る。
- 確認できたら`REVIEW_ACTION="accept"`で記録し、02またはCLIでfreezeする。Test正解を見て設定を調整しない。


In [ ]:
if REVIEW_ACTION is not None:
    if selected_context is None or REVIEW_ACTION not in {"accept", "reject"}:
        raise ValueError("02の選択artifactとaccept/rejectの明示が必要です。")
    display(review_selection(WORKFLOW, accepted=REVIEW_ACTION == "accept", note=REVIEW_NOTE))
print("Next:", workflow_state(WORKFLOW)["next_action"])